# Modeling Readiness Assessor — Synthetic Data Provisioner

Creates a reproducible demo workspace with deliberate modeling debt:
- 3 semantic models: CRM-Sales, ERP-Finance, Invoicing-Legacy
- 1 Fabric IQ ontology: Manufacturing-Ontology (Customer + Product = FLL gaps)
- Vendor entity with full source attribution (positive example)

**Safety gate**: Requires `demo_workspace: true` in narrator.config.yaml.
**Idempotent**: Checks for existing resources before creating.

In [ ]:
import yaml, sys, os
from pathlib import Path

CONFIG_PATH = Path('/lakehouse/default/Files/narrator.config.yaml')
if not CONFIG_PATH.exists():
    CONFIG_PATH = Path('narrator.config.yaml')

config = yaml.safe_load(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}

if not config.get('demo_workspace', False):
    raise RuntimeError(
        "SAFETY GATE: demo_workspace is not set to true in narrator.config.yaml. "
        "Set demo_workspace: true before running the provisioner."
    )

print('✓ demo_workspace: true confirmed — provisioner may proceed')
print(f'  Config: {CONFIG_PATH}')

In [ ]:
# Explicit user confirmation before any resource creation
confirm = input("Type 'yes' to create demo resources in this workspace, or anything else to abort: ")
if confirm.strip().lower() != 'yes':
    print('Aborted — no resources were created.')
    sys.exit(0)
print('✓ User confirmed — beginning provisioning...')

In [ ]:
# Cell 3: Create semantic models via Fabric/Power BI REST API
# Uses notebookutils (Fabric built-in) — not a pyproject.toml dependency
import json, time
import requests

def get_token(scope='pbi'):
    return notebookutils.credentials.getToken(scope)  # noqa: F821 — Fabric built-in

def get_workspace_id():
    return notebookutils.runtime.context['currentWorkspaceId']  # noqa: F821

PBI_API = 'https://api.powerbi.com/v1.0/myorg'

DEMO_MODELS = [
    {
        'name': 'CRM-Sales',
        'tables': [{
            'name': 'Customer',
            'columns': [
                {'name': 'CustomerGUID', 'dataType': 'String'},
                {'name': 'CustomerName', 'dataType': 'String'},
                {'name': 'Region', 'dataType': 'String'},
            ]
        }]
    },
    {
        'name': 'ERP-Finance',
        'tables': [{
            'name': 'Account',
            'columns': [
                {'name': 'AccountNumber', 'dataType': 'String'},
                {'name': 'AccountName', 'dataType': 'String'},
                {'name': 'CostCenter', 'dataType': 'String'},
            ]
        }]
    },
    {
        'name': 'Invoicing-Legacy',
        'tables': [{
            'name': 'Customer',
            'columns': [
                {'name': 'InvoiceCustomerID', 'dataType': 'String'},
                {'name': 'BillingName', 'dataType': 'String'},
                {'name': 'PaymentTerms', 'dataType': 'String'},
            ]
        }]
    },
]

workspace_id = get_workspace_id()
token = get_token('pbi')
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

# Fetch existing datasets
existing_resp = requests.get(f'{PBI_API}/groups/{workspace_id}/datasets', headers=headers)
existing_names = {d['name'] for d in existing_resp.json().get('value', [])}
print(f'Existing models: {existing_names or "(none)"}')

created = []
for model_def in DEMO_MODELS:
    if model_def['name'] in existing_names:
        print(f'  – {model_def["name"]}: already exists, skipping')
        continue
    resp = requests.post(
        f'{PBI_API}/groups/{workspace_id}/datasets',
        headers=headers,
        json={'name': model_def['name'], 'tables': model_def['tables']},
    )
    if resp.status_code in (200, 201):
        print(f'  ✓ {model_def["name"]}: created (ID: {resp.json().get("id", "?")})')
        created.append(model_def['name'])
    else:
        print(f'  ✗ {model_def["name"]}: failed ({resp.status_code}) — {resp.text[:200]}')

print(f'Semantic models done: {len(created)} created, {len(DEMO_MODELS) - len(created)} skipped')

In [ ]:
# Cell 4: Create Fabric IQ ontology via preview API
# Customer + Product: zero source-attribution properties (deliberate FLL debt)
# Vendor: full source attribution (positive example)

FABRIC_API = 'https://api.fabric.microsoft.com/v1'
onto_token = get_token('fabric')
onto_headers = {'Authorization': f'Bearer {onto_token}', 'Content-Type': 'application/json'}

# Check if ontology already exists
existing_onto = requests.get(
    f'{FABRIC_API}/workspaces/{workspace_id}/ontologies',
    headers=onto_headers,
)
existing_onto_names = {o['name'] for o in existing_onto.json().get('value', [])}

ONTOLOGY_NAME = 'Manufacturing-Ontology'

if ONTOLOGY_NAME in existing_onto_names:
    print(f'  – {ONTOLOGY_NAME}: already exists, skipping')
else:
    ontology_body = {
        'name': ONTOLOGY_NAME,
        'entityTypes': [
            {
                'name': 'Customer',
                'properties': [
                    {'name': 'customer_id', 'dataType': 'String'},
                    {'name': 'customer_name', 'dataType': 'String'},
                ]
                # Deliberately missing: source_system_identifier, source_record_identifier,
                # extraction_timestamp, confidence_score — deliberate FLL debt
            },
            {
                'name': 'Product',
                'properties': [
                    {'name': 'product_id', 'dataType': 'String'},
                    {'name': 'product_name', 'dataType': 'String'},
                ]
                # Deliberately missing all attribution props
            },
            {
                'name': 'Vendor',
                'properties': [
                    {'name': 'vendor_id', 'dataType': 'String'},
                    {'name': 'vendor_name', 'dataType': 'String'},
                    {'name': 'source_system_identifier', 'dataType': 'String', 'isSourceAttribution': True},
                    {'name': 'source_record_identifier', 'dataType': 'String', 'isSourceAttribution': True},
                    {'name': 'extraction_timestamp', 'dataType': 'DateTime', 'isSourceAttribution': True},
                    {'name': 'confidence_score', 'dataType': 'Double', 'isSourceAttribution': True},
                ]
                # Vendor: positive example — full attribution
            },
        ]
    }

    resp = requests.post(
        f'{FABRIC_API}/workspaces/{workspace_id}/ontologies',
        headers=onto_headers,
        json=ontology_body,
    )
    if resp.status_code in (200, 201):
        print(f'  ✓ {ONTOLOGY_NAME}: created')
    else:
        print(f'  ✗ {ONTOLOGY_NAME}: failed ({resp.status_code}) — {resp.text[:200]}')

In [ ]:
# Cell 5: Provisioning summary
print()
print('=== Provisioner Summary ===')
print(f'Workspace: {workspace_id}')
print()
print('Resources provisioned:')
print('  Semantic models: CRM-Sales, ERP-Finance, Invoicing-Legacy')
print('  Ontology: Manufacturing-Ontology (Customer, Product, Vendor)')
print()
print('Deliberate modeling debt:')
print('  - Customer entity has 3 conflicting definitions (CEM debt)')
print('  - Customer and Product lack source-attribution properties (FLL debt)')
print('  - Vendor is a positive example with full source attribution')
print()
print('Next step: Run modeling-readiness-scanner.ipynb against this workspace.')
print('Expected: Canonical entity modeling score ≤ 2, field-level lineage score ≤ 1')